In [7]:
import pickle

import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [2]:
from sklearn.pipeline import make_pipeline

In [3]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("green-taxi-duration")

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1779127674679, experiment_id='3', last_update_time=1779127674679, lifecycle_stage='active', name='green-taxi-duration', tags={}, trace_location=None, workspace='default'>

In [4]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def prepare_dictionaries(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [5]:
df_train = read_dataframe('/workspaces/ml-ops/data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('/workspaces/ml-ops/data/green_tripdata_2021-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [8]:
with mlflow.start_run():
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dict_train, y_train)
    y_pred = pipeline.predict(dict_val)

    rmse = root_mean_squared_error(y_pred, y_val)
    print(params, rmse)
    mlflow.log_metric('rmse', rmse)

    mlflow.sklearn.log_model(pipeline, artifact_path="model")

2026/05/18 18:24:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/18 18:24:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 6.7558229919200725
🏃 View run capable-squid-88 at: http://127.0.0.1:5000/#/experiments/3/runs/c2447fa00080401f904568bff0499c32
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [10]:
from mlflow.tracking import MlflowClient


In [11]:
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
RUN_ID = 'c2447fa00080401f904568bff0499c32'

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [17]:
path = client.download_artifacts(run_id=RUN_ID, path='model')

In [19]:
model = mlflow.sklearn.load_model(f'runs:/{RUN_ID}/model')
